# Part 1 - Reaslistic

In the realistic scenario, additional constraints make the problem more complex and closer to real-life conditions. The key differences are:

Limited expansion: Due to space constraints, no facility can expand beyond 20% of its current capacity. This replaces the 120%/+500 slot allowance from Part 1.

Tiered expansion costs: The cost to expand an existing facility increases with the scale of expansion:

0–10% increase: $20,000 + $200 per existing slot (baseline low cost)

10–15% increase: $20,000 + $400 per existing slot (higher cost)

15–20% increase: $20,000 + $1000 per existing slot (highest cost)

In [4]:
from gurobipy import Model, GRB, quicksum
import pandas as pd
import numpy as np
import os

income_df = pd.read_csv("./ChildCareDeserts_Data/avg_individual_income.csv")
employment_df = pd.read_csv("./ChildCareDeserts_Data/employment_rate.csv")
population_df = pd.read_csv("./ChildCareDeserts_Data/population.csv")
childcare_df = pd.read_csv("./ChildCareDeserts_Data/child_care_regulated.csv")
childcare_df = pd.read_csv("./ChildCareDeserts_Data/potential_locations.csv")


# ---------- Helpers ----------
def normalize_zip(df):
    for col in df.columns:
        if "zip" in col.lower():
            df = df.rename(columns={col: "ZIP"})
            break
    df["ZIP"] = df["ZIP"].astype(str).str.zfill(5)
    return df


def first_col(df, keys):
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in keys):
            return c
    return None


# ---------- normalize zips ----------
income_df = normalize_zip(income_df)
employment_df = normalize_zip(employment_df)
population_df = normalize_zip(population_df)
childcare_df = normalize_zip(childcare_df)
childcare_df = normalize_zip(childcare_df)

# ---------- current capacity per facility & per ZIP ----------
if "total_capacity" in childcare_df.columns:
    childcare_df["fac_capacity"] = childcare_df["total_capacity"]
else:
    cap_cols = [c for c in childcare_df.columns if "capacity" in c.lower()]
    if len(cap_cols) == 0:
        raise ValueError("No capacity columns found in child_care_regulated.csv")
    childcare_df["fac_capacity"] = childcare_df[cap_cols].sum(axis=1, numeric_only=True)
u5_cols = [
    c
    for c in childcare_df.columns
    if any(k in c.lower() for k in ["0-5", "0–5", "infant", "toddler", "preschool"])
]
if len(u5_cols) > 0:
    childcare_df["fac_u5_capacity"] = childcare_df[u5_cols].sum(
        axis=1, numeric_only=True
    )
else:
    childcare_df["fac_u5_capacity"] = 0.0
zip_capacity = (
    childcare_df.groupby("ZIP", as_index=False)["fac_capacity"]
    .sum()
    .rename(columns={"fac_capacity": "current_capacity"})
)
zip_u5_capacity = (
    childcare_df.groupby("ZIP", as_index=False)["fac_u5_capacity"]
    .sum()
    .rename(columns={"fac_u5_capacity": "current_u5_capacity"})
)
# ---------- population (0–5 and 0–12) ----------
c_0_5 = [
    c
    for c in population_df.columns
    if "0-5" in c.lower() or "0–5" in c.lower() or "under 5" in c.lower()
]
c_5_9 = [c for c in population_df.columns if "5-9" in c.lower() or "5–9" in c.lower()]
c_10_14 = [
    c for c in population_df.columns if "10-14" in c.lower() or "10–14" in c.lower()
]


def safe_sum_cols(df, cols):
    return df[cols].sum(axis=1, numeric_only=True) if len(cols) > 0 else 0.0


population_df["pop_0_5"] = safe_sum_cols(population_df, c_0_5)
population_df["pop_5_9"] = safe_sum_cols(population_df, c_5_9)
population_df["pop_10_14"] = safe_sum_cols(population_df, c_10_14)

# 0–12 = (0–5) + (5–9) + (2/5)*(10–14)
population_df["pop_0_12"] = (
    population_df["pop_0_5"]
    + population_df["pop_5_9"]
    + 0.4 * population_df["pop_10_14"]
)

zip_population = population_df[["ZIP", "pop_0_5", "pop_0_12"]].copy()
# ---------- employment / income ----------
employment_df = employment_df.rename(columns={"employment rate": "employment_rate"})
income_df = income_df.rename(columns={"average income": "average_income"})

# ---------- per-ZIP base table ----------
zip_df = childcare_df[["ZIP"]].drop_duplicates()
zip_df = zip_df.merge(zip_capacity, on="ZIP", how="left")
zip_df = zip_df.merge(zip_u5_capacity, on="ZIP", how="left")
zip_df = zip_df.merge(zip_population, on="ZIP", how="left")
zip_df = zip_df.merge(employment_df[["ZIP", "employment_rate"]], on="ZIP", how="left")
zip_df = zip_df.merge(income_df[["ZIP", "average_income"]], on="ZIP", how="left")
zip_df[
    [
        "current_capacity",
        "current_u5_capacity",
        "pop_0_5",
        "pop_0_12",
        "employment_rate",
        "average_income",
    ]
] = zip_df[
    [
        "current_capacity",
        "current_u5_capacity",
        "pop_0_5",
        "pop_0_12",
        "employment_rate",
        "average_income",
    ]
].fillna(0)

ZIPs = zip_df["ZIP"].tolist()
# ---------- facility options (Problem 1) ----------
facility_options = [
    {"name": "S", "cap_total": 100, "cap_05_max": 50, "cost": 65000},
    {"name": "M", "cap_total": 200, "cap_05_max": 100, "cost": 95000},
    {"name": "L", "cap_total": 400, "cap_05_max": 200, "cost": 115000},
]
# ---------- MODEL ----------
m = Model("MinCostChildcare_P1")

# --- NEW FACILITIES: integer count per ZIP and size ---
n_build = {
    (z, opt["name"]): m.addVar(vtype=GRB.INTEGER, lb=0, name=f"build_{z}_{opt['name']}")
    for z in ZIPs
    for opt in facility_options
}

# Total new capacity and new under-5 capacity per ZIP
new_total = {
    z: quicksum(
        n_build[(z, opt["name"])] * opt["cap_total"] for opt in facility_options
    )
    for z in ZIPs
}
new_u5 = {z: m.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"new_u5_{z}") for z in ZIPs}
for z in ZIPs:
    cap_u5_max = quicksum(
        n_build[(z, opt["name"])] * opt["cap_05_max"] for opt in facility_options
    )
    m.addConstr(new_u5[z] <= cap_u5_max, name=f"u5_new_cap_{z}")

# --- EXPANSION: per existing facility f (cap min(20%·n_f, 500)) ---
childcare_df = (
    childcare_df.reset_index(drop=True).reset_index().rename(columns={"index": "f_id"})
)
F = childcare_df["f_id"].tolist()
fac_zip = dict(zip(childcare_df["f_id"], childcare_df["ZIP"]))
n_f = dict(zip(childcare_df["f_id"], childcare_df["fac_capacity"]))

# ---------- EXPANSION ----------
t1, t2, t3, x_f = {}, {}, {}, {}

for f in F:
    caplim = 0.20 * n_f[f]

    t1_ub = 0.10 * n_f[f]
    t2_ub = 0.05 * n_f[f]
    t3_ub = 0.05 * n_f[f]

    t1[f] = m.addVar(lb=0, ub=t1_ub, vtype=GRB.CONTINUOUS, name=f"t1_{f}")
    t2[f] = m.addVar(lb=0, ub=t2_ub, vtype=GRB.CONTINUOUS, name=f"t2_{f}")
    t3[f] = m.addVar(lb=0, ub=t3_ub, vtype=GRB.CONTINUOUS, name=f"t3_{f}")
    x_f[f] = m.addVar(lb=0, ub=caplim, vtype=GRB.CONTINUOUS, name=f"x_{f}")

    m.addConstr(x_f[f] == t1[f] + t2[f] + t3[f], name=f"x_sum_{f}")

# aggregate expansion by ZIP
exp_total = {z: quicksum(x_f[f] for f in F if fac_zip[f] == z) for z in ZIPs}

# allow some expansion to count toward 0–5 (if you prefer: set exp_u5 = 0 to be conservative)
exp_u5 = {z: m.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"exp_u5_{z}") for z in ZIPs}
for z in ZIPs:
    m.addConstr(exp_u5[z] <= exp_total[z], name=f"exp_u5_le_exp_{z}")

# ---------- COVERAGE CONSTRAINTS ----------
EMPLOYMENT_T = 60
INCOME_T = 60000

zip_idx = {z: i for i, z in enumerate(ZIPs)}
zip_series = zip_df.set_index("ZIP")

for z in ZIPs:
    employ_rate = float(zip_series.loc[z, "employment_rate"])
    avg_income = float(zip_series.loc[z, "average_income"])
    pop_012 = float(zip_series.loc[z, "pop_0_12"])
    pop_05 = float(zip_series.loc[z, "pop_0_5"])
    curr_total = float(zip_series.loc[z, "current_capacity"])
    curr_u5 = float(zip_series.loc[z, "current_u5_capacity"])

    is_high = (employ_rate >= EMPLOYMENT_T) or (avg_income <= INCOME_T)
    required_total = 0.5 * pop_012 if is_high else (1.0 / 3.0) * pop_012

    # Total coverage (0–12)
    m.addConstr(
        curr_total + exp_total[z] + new_total[z] >= required_total,
        name=f"cov_total_{z}",
    )

    # 0–5 coverage (≥ 2/3 of 0–5 population)
    m.addConstr(
        curr_u5 + exp_u5[z] + new_u5[z] >= (2.0 / 3.0) * pop_05, name=f"cov_u5_{z}"
    )
# ---------- COSTS ----------

expansion_cost_terms = []

for f in F:
    cap = n_f[f]
    if cap <= 0:
        continue
    expansion_cost_terms.append(
        (20000.0 + 200.0 * cap) * (t1[f] / cap)
        + (20000.0 + 400.0 * cap) * (t2[f] / cap)
        + (20000.0 + 1000.0 * cap) * (t3[f] / cap)
    )
expansion_cost = quicksum(expansion_cost_terms)


build_cost = quicksum(
    n_build[(z, opt["name"])] * opt["cost"] for z in ZIPs for opt in facility_options
)

equip_cost = quicksum(100.0 * (new_u5[z] + exp_u5[z]) for z in ZIPs)

m.setObjective(expansion_cost + build_cost + equip_cost, GRB.MINIMIZE)
# ---------- SOLVE ----------
m.optimize()

if m.status == GRB.OPTIMAL:
    print(f"\nOptimal total cost: ${m.objVal:,.2f}\n")

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G231)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 24220 rows, 73186 columns and 114278 nonzeros
Model fingerprint: 0x000b4c9f
Variable types: 66724 continuous, 6462 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+02]
  Objective range  [1e+02, 1e+05]
  Bounds range     [2e-01, 2e+02]
  RHS range        [3e-01, 6e+03]
Found heuristic solution: objective 5.818821e+08
Presolve removed 24219 rows and 73072 columns
Presolve time: 0.22s
Presolved: 1 rows, 114 columns, 114 nonzeros
Found heuristic solution: objective 5.194033e+08
Variable types: 111 continuous, 3 integer (0 binary)

Root relaxation: objective 5.192024e+08, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time
